<a href="https://colab.research.google.com/github/Squad-Nina-da-Hora/wmc-desafio-carros-usados/blob/main/analise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# **Análise de Venda em Empresa de Veículos Usados**
---


🎯 **Objetivo específico:** Entender quais fatores mais impactam no preço de venda.

🎯 **Objetivo geral:** Identificar as variáveis mais relevantes e propor uma análise baseada em correlações e modelos preditivos simples.

---


Desafio Estatística com Python - Correlação e Regressão

Squad Nina da Hora | Bootcamp Data Analytics 2026.1

## 1. Configurações Iniciais

Variáveis da base de dados:

- `id`: Identificador único do veículo
- `make`: Marca do carro (ex: Ford, Toyota)
- `model`: Modelo do carro
- `year`: Ano de fabricação
- `price`: Preço de venda do carro
- `mileage`: Quilometragem (km rodados)
- `engine_size`: Tamanho do motor (em litros)
- `fuel_type`: Tipo de combustível (gasolina, diesel, elétrico)
- `transmission`: Tipo de transmissão (manual, automática)
- `doors`: Número de portas
- `color`: Cor do carro
- `tax`: Taxa anual de imposto veicular
- `mpg`: Milhas por Galão(indicador de eficiência de combustível)
- `sold_date`: Data de venda do veículo

In [ ]:
# ==============================
# IMPORTACOES
# ==============================

import math  # Para funcoes matematicas, como calculo de tamanho de amostra
import numpy as np  # Para operacoes numericas e arrays
import pandas as pd  # Para manipulaco e analise de data frames
from IPython.display import display, Markdown  # Para exibir outputs formatados

# Bibliotecas para criacao de graficos
import seaborn as sns
import matplotlib.pyplot as plt

# Bibliotecas para testes estatisticos e de hipoteses
from scipy import stats
from scipy.stats import shapiro
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


# Carregamento da base de dados
arquivo = 'bd_carros_usados'
url = f'https://raw.githubusercontent.com/Squad-Nina-da-Hora/wmc-desafio-carros-usados/main/{arquivo}.csv'
df = pd.read_csv(url)

# Verifica se a coluna existe antes de tentar remover
if 'id' in df.columns:
  df.drop(columns='id', inplace=True) # Essa coluna nao tem utilidade

df.head()

In [ ]:
# ==============================
# VARIAVEIS PARA REUTILIZACAO
# ==============================

# Armazena a paleta do Seaborn que sera usada em todo o trabalho
paleta = 'flare'
cores = sns.color_palette(paleta, n_colors=2)  # 2 cores fixas

In [ ]:
# ==============================
# PERFIL DO DATASET
# ==============================

linhas = df.shape[0]
colunas = df.shape[1]
print(f"O dataset possui {linhas} linhas e {colunas} colunas.")

display(Markdown("---"))

info_df = pd.DataFrame({
    'Coluna': df.columns,
    'Tipo': df.dtypes.values,
    'Não nulos': df.count().values,
    'Nulos': df.isnull().sum().values,
    'Valores Únicos': df.nunique().values
})

info_df

In [ ]:
# ==============================
# FUNCOES PARA REUTILIZACAO
# ==============================

def estilizar_tabela(dados, paleta=paleta):
  """
  Estiliza a visualizacao do Data Frame
  """
  display(dados.style
          .format('{:.2f}')
          .background_gradient(cmap=sns.color_palette(paleta, as_cmap=True)))


def plotar_heatmap(dados, tamanho=[7,6], titulo=None, **kwargs):
  """
  Gera um grafico de calor (heatmap) estilizado utilizando Seaborn.

  Parametros:
    dados (DataFrame/Series): Matriz de dados ou correlacao a ser plotada.
    tamanho (list): Dimensoes do grafico no formato [largura, altura].
    titulo (str): Titulo que sera exibido no topo do grafico.
    **kwargs: Argumentos adicionais passados diretamente para a funcao sns.heatmap.
  """
  plt.figure(figsize=tamanho)
  plt.title(titulo, fontsize=12, fontweight='bold')
  sns.heatmap(dados, annot=True, cmap=paleta, fmt='.2f', linewidths=0.5, **kwargs)
  plt.tight_layout()
  plt.show()

---
# **PARTE 1: ANÁLISE DE CORRELAÇÃO**
---

 - Correlação entre as variáveis numéricas e o preço do carro (`price`).
 - Quais variáveis estão mais correlacionadas com o preço.
 - Quais estão menos correlacionadas.

In [ ]:
df_numericas = df.select_dtypes(include=['number'])
print(f'Total de variáveis numéricas: {df_numericas.shape[1]}')

# Calcula a correlacao das variaveis numericas
df_corr = df_numericas.corr()
estilizar_tabela(df_corr) # Exibe a tabela estilizada

In [ ]:
# Pega os indexes, ordenando os valores para o menor
col_alvo = 'price'
id_corr_preco = df_corr[col_alvo].sort_values(ascending=False).index

# Calcula a correlacao da coluna 'price' com as colunas selecionadas
df_corr_preco = df_numericas[id_corr_preco].corrwith(df_numericas[col_alvo]).rename(col_alvo).to_frame()
df_corr_preco.round(2).T

---
# **PARTE 2: ANÁLISE DAS 5 VARIÁVEIS MAIS CORRELACIONADAS**
---

In [ ]:
# Criando uma cópia do dataFrame contendo apenas as transformações das variáveis 
# categóricas e o preço

col_cate = ['make', 'model', 'fuel_type', 'transmission', 'color']
df_cat = df[col_cate + ['price']].copy()
df_cat = pd.get_dummies(df_cat, columns=col_cate, dtype=int)

df_cat


In [ ]:
# Mapeamento visual das categorias com a contagem total
# Ao invés de termos 22 novas variáveis, aqui fizemos a transformação das categóricas
# em códigos númericos para compararmos a correlação de apenas mais 5 variáveis
for col in col_cate:
    display(Markdown(f"**Dicionário de Códigos - {col.upper()}:**"))
    
    # Cria a tabela base de categorias e códigos
    cat = df[col].astype('category').cat.categories
    map = pd.DataFrame({
        'Categoria Original': cat,
        'Código Numérico': range(1, len(cat) + 1)
    })
    
    # Calcula o total de aparições de cada categoria no dataset original
    cont_total = df[col].value_counts()
    map['Total de Carros'] = map['Categoria Original'].map(cont_total)
    
    display(map.style.hide(axis='index'))
    display(Markdown("---"))

In [ ]:
# Verificando a correlação das variáveis categóricas 

for col in col_cate:
    # 1. Cria a cópia isolada e converte o texto para o código numérico (+1)
    df_temp = df[[col, 'price']].copy()
    df_temp['codigo_num'] = df_temp[col].astype('category').cat.codes + 1
    
    # 2. Calcula o coeficiente de Spearman contra o preço
    coef = df_temp['price'].corr(df_temp['codigo_num'], method='spearman')
    
    # 3. Exibe os resultados e a checagem na tela
    print(f"\n==================================================")
    print(f"VARIÁVEL: {col.upper()}")
    print(f"Coeficiente de Spearman contra o Preço: {coef:.4f}")
    print(f"--------------------------------------------------")
    print(df_temp[[col, 'codigo_num', 'price']].head(5))

**Para as 5 variáveis com maior correlação com o preço:**
---

 - **Histograma** e **boxplot** de cada variável.
 - **Scatterplot** (gráfico de dispersão), com `price`no eixo **Y** e a variável no eixo **X**

 - **Regressão Linear Simples**, usando as variáveis mais correlacionadas como a variável preditora (**X**) e o preço como variável resposta (**Y**).
  - Interpretar os coeficientes e o R².

# **Extra:**

 - Análise feita a partir do mês de venda dos carros.

In [ ]:
df['sold_date'] = pd.to_datetime(df['sold_date'])
df['sold_month'] = df['sold_date'].dt.month

print(df[['sold_date','sold_month']])
print(sorted(df['sold_month'].value_counts()))

---
# **Conclusão:**
---